In [2]:
import pandas as pd

In [3]:
phishing_path = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_6000_final.csv"
normal_path   = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\normal_6000_final.csv"
test1_path    = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\1차모델_테스트데이터셋.csv"
test2_path    = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\시나리오통화테스트셋.csv"

In [4]:
df_phishing_1 = pd.read_csv(phishing_path)
df_normal_1 = pd.read_csv(normal_path)
df_test1 = pd.read_csv(test1_path)
df_test2 = pd.read_csv(test2_path)


In [5]:
df_phishing_1.head()

,file_id,text,phishing_type,is_phishing
0,phishing_0001,자전거가 아니라 본인 앞으로 된 명의도용 사건 관련 세미나 참석 여부 확인 차 연락...,기관사칭형,1
1,phishing_0002,"김지현 경찰입니다. 어떤 일로 전화하신 건가요? 방금 제가 질문했었는데요, 선생님....",기관사칭형,1
2,phishing_0003,피의자인지 아니면 개인정보가 유출돼 피해를 입으셨는지 확인하려고 연락 드린 겁니다....,기관사칭형,1
3,phishing_0004,"'네, 알겠습니다.' ""안녕하세요, 본인 맞으세요? 네, 서울 중앙지검입니다."" 저...",기관사칭형,1
4,phishing_0005,"안 보셨어요, 업무 보셨나요? 해지하신 시간 말씀해주시겠어요. 시가 예 판 한 예 ...",기관사칭형,1


In [6]:
df_normal_1.head()

,file_id,text,phishing_type,is_phishing
0,normal_0001,"엄마, 저 오늘 정말 신기한 일 있었어요! 제가 오늘 집에 오는 길에 누구를 만났는...",NaN,0
1,normal_0002,오늘 동창회에 다녀왔는데 엄마 말고는 수술 안 해본 사람이 없더라? 정말요? 엄마가...,NaN,0
2,normal_0003,"엄마! 제가 봉사하러 갔던 양로원 할머니께서 티코스터를 선물로 주셨어요! 어머, 티...",NaN,0
3,normal_0004,"엄마! 잘 지내요? 나 요즘 되게 신나요. 응, 엄마는 잘 지내지. 그나저나 우리 ...",NaN,0
4,normal_0005,엄마 저 오늘 정말 신이 나요! 무슨 일이 있었길래 그렇게 신이 났어? 오늘 대학교...,NaN,0


In [7]:
df_test1.head()

,file_name,text,is_phishing
0,0,예 고객님 담당자 김성도 대리입니다.예지금 법무사님이 두분 배정되셨어요.네 네네 네...,1
1,2,6시 되가지고 전화 해봤습니다.예 예 그 앞전에 이면주 법무사님 영수증 확인되셨는데...,1
2,3,네 네 네 여보세요네 네어디에 계시는 겁니까 지금네?지금 어디 계시는 거예요방금 나...,1
3,5,여보세요예 여보세요. 네 감사과장 박철민 과장입니다.예아 OOO고객님 맞으시죠?예아...,1
4,6,신한은행 정부지원자금으로 마이너스 통장 계좌 통합 가능하십니다. 마이너스 통장 발급...,1


In [8]:
df_test2.head()

,file_id,text,is_phishing
0,phishing_1,"네, 여보세요.\n네, 여보세요. 서울지방경찰청 사이퍼 범죄추사팀 박민선 승사관입니...",1
1,phishing_2,"고객님, 안녕하세요. 농협캐피탈입니다.\n고객님은 신용등급 상향 대상자로 선정되셔서...",1
2,phishing_3,"여보세요? 엄마 나야 나 지금 핸드폰 애 저 깨졌어, 친구 폰으로 연락했어.\n누구...",1
3,phishing_4,"네, 여보세요, 고객님, 안녕하세요, 로젠택배입니다, 고객님. 택배가 주소 의류로 ...",1
4,phishing_5,"네, 안녕하세요. 국민건강보험공단입니다. 고객님.\n작년 과오납 환급금이 47만원 ...",1


In [10]:
import pandas as pd
import numpy as np
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm import tqdm
import time
import os

# 데이터 경로
phishing_path = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_6000_final.csv"
normal_path   = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\normal_6000_final.csv"
test1_path    = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\1차모델_테스트데이터셋.csv"
test2_path    = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\시나리오통화테스트셋.csv"

# 1. 훈련 데이터 로드
df_phish  = pd.read_csv(phishing_path)
df_normal = pd.read_csv(normal_path)
df_train  = pd.concat([df_phish, df_normal], ignore_index=True).dropna(subset=["text"])
X_train   = df_train["text"].astype(str).tolist()
y_train   = df_train["is_phishing"].values

# 2. Kiwi 토크나이저 정의
def kiwi_tokenizer(txt: str):
    kiwi = Kiwi()  # 함수 안에서 생성
    toks = []
    for morph in kiwi.analyze(txt)[0][0]:
        if morph.tag in ("NNG", "NNP"):
            toks.append(morph.form)
        elif morph.tag in ("VV", "VA"):
            toks.append(morph.form + "다")
    return toks

# 3. 파이프라인 정의
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=kiwi_tokenizer, lowercase=False)),
    ("clf", LGBMClassifier(objective="binary", metric="binary_logloss",
                          boosting_type="gbdt", random_state=42, n_jobs=-1))
])

# 4. 하이퍼파라미터 그리드
param_grid = {
    "tfidf__max_df":      [0.85, 0.9, 0.95],
    "tfidf__min_df":      [3, 5, 8],
    "tfidf__ngram_range": [(1, 2)],
    "clf__n_estimators":  [100, 200, 300],
    "clf__learning_rate": [0.05, 0.1, 0.15],
}

# 5. 교차검증 설정
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 6. GridSearchCV 실행
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

print("하이퍼파라미터 튜닝 시작...")
start_time = time.time()
grid.fit(X_train, y_train)
print(f"튜닝 완료 (소요 {(time.time() - start_time)/60:.1f}분)")
print(f"최적 F1-score: {grid.best_score_:.4f}")
print("최적 파라미터:", grid.best_params_)

# 7. 최적 모델 저장
best_model = grid.best_estimator_
model_save_path = "phishing_detection_lightgbm_model.pkl"
joblib.dump(best_model, model_save_path)
print(f"모델 저장: {model_save_path} ({os.path.getsize(model_save_path)/1024/1024:.2f} MB)")

# 8. 테스트 데이터 평가 함수
def evaluate_test_data(pipeline, test_path, test_name):
    df = pd.read_csv(test_path).dropna(subset=["text", "is_phishing"])
    X = df["text"].astype(str).tolist()
    y = df["is_phishing"].values

    print(f"\n=== {test_name} 평가 ===")
    print(f"샘플 수: {len(y)}  레이블 분포: {df['is_phishing'].value_counts().to_dict()}")

    # 예측
    y_pred     = []
    y_pred_proba = []
    for i in tqdm(range(0, len(X), 100), desc="예측 중"):
        batch = X[i:i+100]
        y_pred.extend(pipeline.predict(batch))
        y_pred_proba.extend(pipeline.predict_proba(batch)[:,1])

    # 지표 계산
    acc  = accuracy_score(y, y_pred)
    pre  = precision_score(y, y_pred, zero_division=0)
    rec  = recall_score(y, y_pred, zero_division=0)
    f1   = f1_score(y, y_pred, zero_division=0)
    cm   = confusion_matrix(y, y_pred)
    
    print(f"Accuracy: {acc:.4f}  Precision: {pre:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y, y_pred, target_names=["정상","피싱"]))

    # 혼동행렬 시각화
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["정상","피싱"], yticklabels=["정상","피싱"])
    plt.title(f"{test_name} 혼동 행렬")
    plt.xlabel("예측")
    plt.ylabel("실제")
    plt.tight_layout()
    plt.show()

    return {"accuracy":acc, "precision":pre, "recall":rec, "f1":f1, "cm":cm, "n":len(y)}

# 9. 테스트셋 평가
results1 = evaluate_test_data(best_model, test1_path, "1차모델 테스트데이터셋")
results2 = evaluate_test_data(best_model, test2_path, "시나리오통화 테스트셋")

# 10. 전체 요약
summary = pd.DataFrame([
    ["1차모델 테스트데이터셋", results1["n"], results1["accuracy"], results1["precision"], results1["recall"], results1["f1"]],
    ["시나리오통화 테스트셋",   results2["n"], results2["accuracy"], results2["precision"], results2["recall"], results2["f1"]],
], columns=["테스트셋","샘플수","Accuracy","Precision","Recall","F1-Score"])
print("\n=== 전체 테스트 결과 요약 ===")
print(summary.round(4).to_string(index=False))

# 11. 성능 비교 시각화
fig, axes = plt.subplots(1,2, figsize=(14,5))
metrics = ["Accuracy","Precision","Recall","F1-Score"]
x = np.arange(len(metrics))
width = 0.35

scores1 = [results1[m.lower()] for m in metrics]
scores2 = [results2[m.lower()] for m in metrics]

axes[0].bar(x-width/2, scores1, width, label="Test1")
axes[0].bar(x+width/2, scores2, width, label="Test2")
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0,1); axes[0].set_title("성능 비교"); axes[0].legend()

for i,v in enumerate(scores1):
    axes[0].text(i-width/2, v+0.01, f"{v:.3f}", ha="center")
for i,v in enumerate(scores2):
    axes[0].text(i+width/2, v+0.01, f"{v:.3f}", ha="center")

axes[1].bar(["Test1","Test2"], [results1["n"], results2["n"]], color=["skyblue","lightcoral"])
axes[1].set_title("샘플 수 비교")
axes[1].set_ylabel("샘플 수")
for i,v in enumerate([results1["n"], results2["n"]]):
    axes[1].text(i, v+len([results1["n"], results2["n"]])*0.01, str(v), ha="center")

plt.tight_layout()
plt.show()

하이퍼파라미터 튜닝 시작...
Fitting 5 folds for each of 81 candidates, totalling 405 fits


KeyboardInterrupt: 